In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *
from textblob import TextBlob

#pd.set_option("display.max_colwidth', -1)
from time import time
import re
import string
import os

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata

In [ ]:
# %run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/Sentiment/Functions.ipynb"
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDate.pkl', compression=None)
df_companies=pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SPXCompaniesByMarketCap.pkl")

In [ ]:
df_articles.index = df_articles["Date"]


In [ ]:
df_articles.loc[df_articles["Date"]=="2021-12-27"]#.sum()#.groupby("PressReleaseCount").count()

In [ ]:
df_companies

In [ ]:
# df_companies.reset_index(inplace=True)#.loc[:"2020-06-01"].tail(50)#.groupby("date").sum().plot()

In [ ]:
df_companies["marketCap_ln_return"]=np.log(df_companies["marketCap"]/df_companies["marketCap"].shift(periods=1))

In [ ]:
df_companies.dropna(inplace=True)

In [ ]:
df_companies['marketCap_ln_return'].dropna(inplace=True)#.max()
df_companies = df_companies.loc[df_companies['marketCap_ln_return'] != np.inf]
df_companies = df_companies.loc[df_companies['marketCap_ln_return'] != -np.inf]

In [ ]:
# New words and values
new_words = {
    'crushes': 100,
    'beats': 5,
    'misses': -5,
    'trouble': -10,
    'falls': -100,
    "dividends":100,
    "dividend":100,
    "earning":100,
    "ipo":100,
    "alliance":100,
    "enhance":100,
    "enhanc":100
}
# Instantiate the sentiment intensity analyzer with the existing lexicon
vader = SentimentIntensityAnalyzer()
# Update the lexicon
vader.lexicon.update(new_words)

In [ ]:
# df_companies.groupby("date").sum().loc[2]#loc[df_companies.index.get_level_values("date")==date].values.sum()

In [ ]:
def MarketCapFactor(companies,date):
    marketcap = 0
    for company in companies:
        try:
            # print(df_companies.loc[company, date])
            marketcap += df_companies.loc[company, date].values[0]
        except:
            print("Exception: MarketCapFactor")
            # print(company)
            # print(date)
            # print("===============")
    totalMarketCap = df_companies.loc[df_companies.index.get_level_values("date")==date].values.sum()
    MarketCap_Factor = (marketcap/totalMarketCap)
    return MarketCap_Factor

In [ ]:
df_companies

In [ ]:
df_companies.loc[df_companies.index.get_level_values("date")=="2020-10-27"]

In [ ]:
df_companies

In [ ]:
# df_companies.loc["RTX", "2020-10-27"].values[0]

In [ ]:
# # https://stackoverflow.com/questions/20290870/improving-the-extraction-of-human-names-with-nltk
# import nltk
# from nltk.tag.stanford import NERTagger
# st = NERTagger('stanford-ner/all.3class.distsim.crf.ser.gz', 'stanford-ner/stanford-ner.jar')
# text = "Farid Srouji"

# for sent in nltk.sent_tokenize(text):
#     tokens = nltk.tokenize.word_tokenize(sent)
#     tags = st.tag(tokens)
#     for tag in tags:
#         if tag[1]=='PERSON': print (tag)

In [ ]:
with open('/Users/Farid Srouji/Documents/Research/master/globaldata/companylist.txt') as f:
    companies = f.read().splitlines()

# 2) HOUR

In [ ]:
# Reading the saved data pickle file
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDateHourDF.pkl', compression=None)
# del df 

## Vader

In [ ]:
debug=False
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0

rows_list = []
for date, row in df_articles.T.iteritems():
    Hour = int(row["Hour"])
    print(Hour)
    sentence = unicodedata.normalize('NFKD', row['Articles'])#.encode('ascii','ignore').decode('ascii')
    sentence = str(sentence)
    sentence = sentence.strip("[]")
    sentence = sentence.strip("''")

    prc = row['PressReleaseCount']
    ss = vader.polarity_scores(sentence)

    Compound = ss['compound']
    Neg = ss['neg']
    Neu = ss['neu']
    Pos = ss['pos']
    companieslist = list(filter(None, row['Company'].split(". ")))
    mcf = MarketCapFactor(companieslist, date)
    Sentiment=0
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            Sentiment = ((100*(ss['neg'] / (ss['pos'] + ss['neg'])))*(prc/mean_prc))*mcf

    countNeg+=ss['neg']
    countNeu+=ss['neu']
    countPos+=ss['pos']
    
    ############## TEXTBLOB ##############
    blob = TextBlob(sentence)
    print(date)
    Polarity = blob.sentiment.polarity
    Subjectivity = blob.sentiment.subjectivity
    polarityB = ""
    if (blob.sentiment.polarity > 0 ):
        polarityB = "Positive"
    elif (blob.sentiment.polarity < 0 ):
        polarityB = "Negative"
    elif (blob.sentiment.polarity == 0 ):
        polarityB = "Neutral"

    new_row = {"Date":date, "Hour": Hour, "compound":Compound, "neg":Neg, "neu":Neu, "pos":Pos, "sentiment":Sentiment, "polarity": Polarity, "subjectivity":Subjectivity, "polarityB":polarityB,"Articles":sentence, "Title":row["Title"], "Text":row["Text"], "Company":row["Company"], "PressReleaseCount":row["PressReleaseCount"], "MarketCapFactor":MarketCapFactor}
    rows_list.append(new_row)
df = pd.DataFrame(data=rows_list, columns=["Date", "Hour", "compound", "neg", "neu", "pos", "sentiment", "polarity", "subjectivity", "polarityB","Articles","Title","Text", "Company", "PressReleaseCount", "MarketCapFactor"])
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentHourly.pkl")

In [ ]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

In [ ]:
del df

# 3) TRADING HOURS 09-16

In [ ]:
# Reading the saved data pickle file
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDate_09_15.pkl', compression=None)

In [ ]:
df_articles

In [ ]:
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['sentiment']] = ''

In [ ]:
debug=False
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0

for date, row in df_articles.T.iteritems():
    # print("The date is:"+str(date))
    # try:
    print(type(df_articles.loc[date, 'Articles']))
    sentence = unicodedata.normalize('NFKD', row['Articles'])#.encode('ascii','ignore').decode('ascii')
    sentence = str(preprocessing_text(sentence))
    sentence = sentence.strip("[]")
    sentence = sentence.strip("''")
    df.at[date, 'Articles'] = sentence
    prc = row['PressReleaseCount']

    ss = vader.polarity_scores(sentence)

    df.at[date, 'compound'] = ss['compound']
    df.at[date, 'neg'] = ss['neg']
    df.at[date, 'neu'] = ss['neu']
    df.at[date, 'pos'] = ss['pos']
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            df.at[date, 'sentiment'] = ((100*(ss['neg'] / (ss['pos'] + ss['neg'])))*(prc/mean_prc))

    countNeg+=ss['neg']
    countNeu+=ss['neu']
    countPos+=ss['pos']

In [ ]:
df_articles.drop(columns={"Articles"},inplace=True)
df = df.join(df_articles)

In [ ]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment_09_15.pkl")

In [ ]:
del df

# 4) Not Trading Hours 16-09

In [ ]:
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDate_16_08.pkl', compression=None)

In [ ]:
df_articles

In [ ]:
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['sentiment']] = ''

In [ ]:
debug=False
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0

for date, row in df_articles.T.iteritems():
    # print("The date is:"+str(date))
    # try:
    print(type(df_articles.loc[date, 'Articles']))
    sentence = unicodedata.normalize('NFKD', df_articles.loc[date, 'Articles'])#.encode('ascii','ignore').decode('ascii')
    sentence = str(preprocessing_text(sentence))
    sentence = sentence.strip("[]")
    sentence = sentence.strip("''")
    df.at[date, 'Articles'] = sentence
    prc = row['PressReleaseCount']

    ss = vader.polarity_scores(sentence)

    #print(sentence)
    #print("#2")
    df.at[date, 'compound'] = ss['compound']
    df.at[date, 'neg'] = ss['neg']
    df.at[date, 'neu'] = ss['neu']
    df.at[date, 'pos'] = ss['pos']
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            df.at[date, 'sentiment'] = ((100*(ss['neg'] / (ss['pos'] + ss['neg'])))*(prc/mean_prc))

    countNeg+=ss['neg']
    countNeu+=ss['neu']
    countPos+=ss['pos']

In [ ]:
df_articles.drop(columns={"Articles"},inplace=True)
df = df.join(df_articles)

In [ ]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment_16_08.pkl")